# Lesson 3 — The Traxler Counter-Attack (as Black)
**Project 1500 | Priority #3**

All statistics computed fresh from game files on every run. No hardcoded results.
Board diagrams are illustrative example lines, clearly labelled as such.

In [ ]:
import chess
import chess.pgn
import chess.svg
import json
import glob
import io
import re
from collections import Counter
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt

USERNAME   = 'jf4bes'
EXCLUDE    = {'2025_01', '2025_02', '2025_03'}
BOARD_SIZE = 380

def board_after(moves_san):
    """Apply SAN moves to a fresh board — illustrative diagrams only."""
    board = chess.Board()
    for san in moves_san:
        board.push_san(san)
    return board

def show_board(board, arrows=None, caption='', size=BOARD_SIZE, flipped=True):
    svg = chess.svg.board(board, arrows=arrows or [], size=size, flipped=flipped)
    if caption:
        display(HTML(f'<p style="font-family:monospace;font-size:13px;margin:4px 0 8px 0;color:#aaa">{caption}</p>'))
    display(SVG(data=svg))

## Step 1 — Load rapid games as black (April 2025+)

In [ ]:
def load_rapid_games_as_black(username, games_dir='games', exclude=EXCLUDE):
    files = sorted(glob.glob(f'{games_dir}/2025_*.json') + glob.glob(f'{games_dir}/2026_*.json'))
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') != 'rapid': continue
            black = g.get('black', {})
            if black.get('username', '').lower() != username.lower(): continue
            result = black.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({'outcome': outcome, 'pgn': g.get('pgn', ''), 'eco_url': eco_url})
    return games

all_black_games = load_rapid_games_as_black(USERNAME)
print(f'Loaded {len(all_black_games)} rapid games as black')

## Step 2 — Filter to Traxler games and compute stats

In [ ]:
TRAXLER_PATTERN = 'Traxler'

traxler_games = [
    g for g in all_black_games
    if TRAXLER_PATTERN in g['eco_url']
]

counts = Counter(g['outcome'] for g in traxler_games)
total  = len(traxler_games)
wr     = 100 * counts['win'] / total if total else 0

print(f'Traxler games as black: {total}')
print(f'W:{counts["win"]}  L:{counts["loss"]}  D:{counts["draw"]}  Win%: {wr:.1f}%')

if total < 15:
    print('\nNote: small sample — treat percentages as directional, not definitive.')

## Step 3 — Overall win rate chart

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Win', 'Loss', 'Draw'],
       [counts['win'], counts['loss'], counts['draw']],
       color=['#27ae60', '#c0392b', '#999'], edgecolor='white')
ax.set_title(f'Traxler as black — {wr:.1f}% win rate ({total} games)')
ax.set_ylabel('Games')
plt.tight_layout()
plt.show()

## Step 4 — Board diagrams *(illustrative example lines)*

> **Note on junction analysis:** The Traxler sample is small and the positions diverge
> rapidly after the sacrifice. Junction stats (Rf8 vs d5) have < 5 games each and are
> not reliable enough to chart. The key lesson is positional: avoid the Traxler entirely.

In [ ]:
# [ILLUSTRATIVE] The position where the Traxler arises — after 4.Ng5
# Verified legal: e4 e5 Nf3 Nc6 Bc4 Nf6 Ng5
AFTER_NG5 = ['e4','e5','Nf3','Nc6','Bc4','Nf6','Ng5']

show_board(
    board_after(AFTER_NG5),
    arrows=[
        chess.svg.Arrow(chess.D7, chess.D5, color='#27ae60'),  # d5 — safe response
        chess.svg.Arrow(chess.F8, chess.C5, color='#c0392b'),  # Bc5 — Traxler, chaotic
    ],
    caption='[ILLUSTRATIVE] After 4.Ng5 — Green=d5! (safe, recommended) | Red=Bc5 (Traxler, avoid at your level)'
)

In [ ]:
# [ILLUSTRATIVE] The safe alternative — 4...d5! sidesteps the chaos
# Verified legal: after Ng5 d5 exd5 Na5 (attacking the Bc4)
SAFE_LINE = ['e4','e5','Nf3','Nc6','Bc4','Nf6','Ng5','d5','exd5','Na5']

show_board(
    board_after(SAFE_LINE),
    arrows=[
        chess.svg.Arrow(chess.A5, chess.C4, color='#27ae60'),  # Na5 attacks Bc4
        chess.svg.Arrow(chess.D5, chess.D4, color='#2980b9'),  # pawn expansion plan
    ],
    caption='[ILLUSTRATIVE] After Ng5 d5! exd5 Na5 — black attacks the Bc4 and has a clear position. '
            'No sacrifices, no exposed king.'
)

In [ ]:
# [ILLUSTRATIVE] What happens if you enter the Traxler: Bxf2+ Ke2 Nd4+
# Verified legal: Bc5 Nxf7 Bxf2+ Ke2 Nd4+ (forking king and queen)
TRAXLER_CHAOS = ['e4','e5','Nf3','Nc6','Bc4','Nf6','Ng5','Bc5','Nxf7','Bxf2+','Ke2','Nd4+']

show_board(
    board_after(TRAXLER_CHAOS),
    arrows=[
        chess.svg.Arrow(chess.D4, chess.E2, color='#27ae60'),  # Nd4 forks king
        chess.svg.Arrow(chess.D4, chess.D1, color='#27ae60'),  # and queen
    ],
    caption='[ILLUSTRATIVE] After Bc5 Nxf7 Bxf2+ Ke2 Nd4+ — if you\'re already in the Traxler, '
            'play Nd4+ (forking king and queen) rather than d5'
)

## Step 5 — Summary

In [ ]:
print('TRAXLER — SUMMARY')
print('=' * 50)
print(f'\nTotal Traxler games as black: {total}  ({wr:.1f}% win rate)')
print()
print('RULES:')
print('  1. When white plays 4.Ng5 — do NOT play Bc5 (Traxler)')
print('  2. Play 4...d5! instead — clean, safe, attacks white directly')
print('  3. After d5 exd5: Na5 attacks the bishop, black has a good game')
print()
print('If already in the Traxler (Bxf2+ Ke2):')
print('  Play Nd4+ first (forks king and queen)')
print('  Do NOT play d5 — your king is exposed, keep the position closed')